# Prepare Differential Inputs QA

Inspection notebook for `20_differential_build_inputs.ipynb` outputs.
This notebook is intentionally QA-only; canonical generation is performed in `20_differential_build_inputs.ipynb`.


ALLOW_CATEGORY_COUNT_MISMATCH = os.getenv("ALLOW_CATEGORY_COUNT_MISMATCH", "false").lower() in {"1", "true", "yes", "on"}


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import yaml

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dx_chat_entropy.paths import get_paths
from dx_chat_entropy.lr_differential_inputs import parse_matrix_sheet

PATHS = get_paths(REPO_ROOT)


ALLOW_CATEGORY_COUNT_MISMATCH = os.getenv("ALLOW_CATEGORY_COUNT_MISMATCH", "false").lower() in {"1", "true", "yes", "on"}


In [ ]:
CONFIG_PATH = REPO_ROOT / "config/lr_differential_scenarios.yaml"
PAIRS_MANIFEST = PATHS.processed / "lr_differential/manifests/pairs_manifest.csv"
RUN_MANIFEST = PATHS.processed / "lr_differential/manifests/run_manifest.json"

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
pd.DataFrame(config["scenarios"])[
    ["scenario_id", "sheet_name", "parser_profile", "expected_category_count"]
]


In [ ]:
summary_rows = []
for scenario in config["scenarios"]:
    wb = REPO_ROOT / scenario["source_workbook"]
    df = pd.read_excel(wb, sheet_name=scenario["sheet_name"], header=None)
    parsed = parse_matrix_sheet(
        df,
        parser_profile=scenario["parser_profile"],
        key_feature_pattern=scenario.get("key_feature_token", "^key feature"),
        expected_category_count=scenario.get("expected_category_count"),
        allow_category_count_mismatch=ALLOW_CATEGORY_COUNT_MISMATCH,
    )
    summary_rows.append(
        {
            "scenario_id": scenario["scenario_id"],
            "observed_categories": len(parsed.categories),
            "findings": len(parsed.findings),
            "warnings": "; ".join(parsed.warnings),
        }
    )

pd.DataFrame(summary_rows)


In [ ]:
if PAIRS_MANIFEST.exists():
    pairs_df = pd.read_csv(PAIRS_MANIFEST)
    print(f"Pairs in manifest: {len(pairs_df)}")
    display(pairs_df.groupby("scenario_id").size().rename("pair_count").to_frame())

    sample_input = REPO_ROOT / pairs_df.iloc[0]["input_workbook"]
    print(f"Sample workbook: {sample_input}")
    display(pd.read_excel(sample_input, header=None).head(8))
else:
    print("Pairs manifest not found. Run notebooks/20_differential_build_inputs.ipynb first.")

if RUN_MANIFEST.exists():
    run_manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))
    print(json.dumps({
        "scenario_count": run_manifest.get("scenario_count"),
        "pair_count": run_manifest.get("pair_count"),
        "warnings": run_manifest.get("warnings", []),
    }, indent=2))
